# GEM TREQ: sitio, licuación y ladera

Análisis exploratorio independiente de bronce. Fecha de revisión: 13 de noviembre de 2026.

Se examinan los archivos locales originales, no una tabla Delta ejecutada en Databricks.
No se limpia ni sobrescribe la fuente. Las transformaciones que aparezcan se hacen en copias
para evaluar lo que necesitará plata. Los resultados no prueban causalidad ni certifican seguridad.

Cada notebook sigue el orden: procedencia y unidad, calidad por columna y geometría,
distribuciones, límites y reglas para plata. No se muestran registros individuales de edificios oficiales.

In [ ]:
from pathlib import Path
import sys, json, re, unicodedata
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "CLAUDE.md").exists() and (p / "data/externos").is_dir())
sys.path.insert(0, str(ROOT / "analisis_bronce"))
from soporte import (ficha, huella, leer_geojson, mostrar_perfil, geometria, conteos,
                     faltantes, guardar_resumen, grafico_barras, SALIDAS, CRS_METROS)
pd.set_option("display.max_columns", 24)
pd.set_option("display.max_rows", 60)
plt.rcParams.update({"font.size": 11, "figure.dpi": 110})

## 1. Procedencia y unidad

Una fila es un sitio de modelos GEM que combina susceptibilidad a licuación y pendiente. Son susceptibilidades/condiciones, no evidencia de que hubo licuación o deslizamiento.

Este archivo fue materializado por `scripts/descargar_externos.py`. Se distingue de los originales remotos: el script recorta, combina o deriva columnas antes de escribirlo.

In [ ]:
PATH=ROOT/'data/externos/gem_sitio_licuacion_ladera.csv'
meta=ficha(PATH)
df=pd.read_csv(PATH,keep_default_na=False)
calidad=mostrar_perfil(df)
x=df.copy()
g=gpd.GeoDataFrame(x,geometry=gpd.points_from_xy(pd.to_numeric(x.lon),pd.to_numeric(x.lat)),crs=4326)
geo=geometria(g)
display(pd.Series({**meta, "filas":len(df),"columnas":len(df.columns)}).to_frame("valor"))
display(pd.Series(geo).to_frame("geometria"))
resumen={"dataset":'14_gem_licuacion_ladera',**meta,"filas":len(df),"columnas":list(df),"calidad_columnas":calidad.to_dict("records"),"geometria":geo}

## 2. Categorías y rangos físicos

In [ ]:
categorias=conteos(df.liq_susc_cat,"susceptibilidad")
display(categorias)
display(df[["gwd","pendiente_grados"]].describe(percentiles=[.01,.5,.95,.99]))
qa={"duplicados_lon_lat":int(df.duplicated(["lon","lat"]).sum()),
 "categorias_no_reconocidas":sorted(set(df.liq_susc_cat)-{"vh","h","m","l","vl","n"}),
 "pendientes_fuera_0_90":int((~df.pendiente_grados.between(0,90)).sum()),
 "gwd_negativo":int(df.gwd.lt(0).sum())}
display(pd.Series(qa).to_frame("control"))
grafico_barras(categorias,"Susceptibilidad modelada a licuación", "14_licuacion")
resumen.update(qa);resumen["categorias"]=df.liq_susc_cat.value_counts().to_dict()
resumen["pendiente"]=df.pendiente_grados.describe().to_dict()

## Reglas decididas para plata

- **Seleccionar como contexto de sitio condicionado a soporte espacial.** Conservar códigos de susceptibilidad y pendiente por separado.
- Documentar el cruce lon/lat entre las dos fuentes originales. El CSV combinado es derivado, no una copia cruda de ambas.
- Mantener `n` (nula según el modelo) separado de faltante. Un mapeo ordinal 0–1 es una decisión de modelado, no probabilidad de ocurrencia.
- Guardar distancia del sitio asignado y una bandera de extrapolación; no inferir afectación observada solo desde susceptibilidad.

In [ ]:
assert huella(PATH) == meta["sha256"], "La fuente cruda cambió durante el análisis"
guardar_resumen('14_gem_licuacion_ladera', resumen)
print("Resumen y gráficos guardados; la fuente cruda conserva su SHA-256.")